# Chapter 6 — Inspect State, Don't Guess

**Book alignment:** Debugging AI From First Principles, Chapter 6

**Question this notebook isolates:** `apply_discount` returns \$160 instead of \$140. The
same wrong output comes from **H1** (coupon table holds 0, branch runs) and **H2** (branch
skipped because `coupon` is `None`). An end-of-function `print` cannot separate them. Does
*predict-then-inspect* at function entry — write the value+type you expect per line, then
let the runtime overrule you — convict the right one in one pass?

In [ ]:
RATE_TABLE = {"premium": 0.2, "standard": 0.1}
COUPONS    = {"SAVE20": 20.0}

def apply_discount(order, *, trace=None):
    def rec(line, **vals):
        if trace is not None:
            trace.append((line, {k: (v, type(v).__name__) for k, v in vals.items()}))
    rate = RATE_TABLE[order["tier"]]
    rec(12, rate=rate)
    total = order["subtotal"] * (1 - rate)
    rec(13, total=total)
    if order.get("coupon"):
        rec(14, coupon=order.get("coupon"), branch="taken")
        total -= COUPONS[order["coupon"]]
        rec(15, total=total)
    else:
        rec(14, coupon=order.get("coupon"), branch="skipped")
    return round(total, 2)

# the caller normalises the order one frame up. a refactor dropped the .upper(),
# so a lowercase coupon misses the table and is silently set to None.
def normalize_order(raw):
    code = raw.get("coupon", "")
    return {**raw, "coupon": code if code in COUPONS else None}

## 1. The symptom: \$160, expected \$140 — and it reads correctly

In [ ]:
order = normalize_order({"tier": "premium", "subtotal": 200, "coupon": "save20"})
got = apply_discount(order)
print(f"apply_discount(...) = {got}   expected 140.0")
assert got == 160.0
# an end-of-function print sees 160.0 under BOTH hypotheses:
#   H1: COUPONS['SAVE20']==0, branch runs -> 200*0.8 - 0 = 160
#   H2: branch skipped         -> 200*0.8       = 160
print("output-only inspection cannot separate H1 (wrong value) from H2 (skipped branch)")

## 2. Predict-then-inspect: write predictions BEFORE stepping

In [ ]:
PREDICTIONS = {
    12: ("rate == 0.2 (float)",              lambda t: t["rate"] == (0.2, "float")),
    14: ("coupon == 'save20' (str, truthy)", lambda t: t["coupon"][0] == "save20"),
    15: ("line 15 executes; total == 140.0", lambda t: t.get("total", (None,))[0] == 140.0),
}

trace = []
apply_discount(order, trace=trace)
seen = {line: vals for line, vals in trace}
print(f"{'line':>4}  {'PREDICTION':40}  OBSERVATION")
first_falsified = None
for line, (desc, check) in PREDICTIONS.items():
    if line in seen:
        ok = check(seen[line])
        obs = seen[line]
    else:
        ok, obs = False, "line never executed"
    verdict = "confirmed" if ok else "FALSIFIED"
    if not ok and first_falsified is None:
        first_falsified = line
    print(f"{line:>4}  {desc:40}  {obs}   [{verdict}]")

assert seen[12]["rate"] == (0.2, "float")               # P12 confirmed - rate is fine
assert 15 not in seen                                   # P15 falsified - branch skipped
assert first_falsified == 14                            # coupon is None at entry, not 'save20'
print(f"\nfirst falsified prediction: line {first_falsified}. H2 (skipped branch) convicted.")

## 3. Step up: the bad value originated in the caller

In [ ]:
print("order at apply_discount entry:", order)
assert order["coupon"] is None
# normalize_order looked up COUPONS.get('SAVE20'-uppercased) but returned the *string or None*
# incorrectly - a refactor replaced `coupon.strip().upper()` with a lookup that yields None on miss
raw = {"tier": "premium", "subtotal": 200, "coupon": "save20"}
print("normalize_order bug: 'save20' -> ", normalize_order(raw)["coupon"], " (should be 'SAVE20')")
print("the first divergence is ABOVE apply_discount - fix the caller, not the discount table")

## What we earned

\$160 instead of \$140 arises identically from a stale table (branch runs) and a skipped
branch (`coupon is None`) — an output print confirms the symptom and localises nothing.
Predict-then-inspect at *entry*, with value **and type** written before looking, falsifies
the line-14 prediction on the first pass: the coupon is `None`, `rate` is correct, line 15
never runs. The fix is one frame up, in the caller that normalised the coupon away.

**Notebook 07 / Chapter 7** hunts the inputs the suite never ran: the boundaries.